# Agentic RAG on local CPU (GGUF int8) / Colab

Replication of MoRAGBench's on-device agentic pipeline (`Agentic.kt` + `agentic_pipeline_config.json`) as one notebook.

Runs on **local CPU** (or Colab) with the official **Qwen GGUF Q8_0 int8** model via llama.cpp — auto-downloaded in `set_model`.

**Pipeline arms** (same `aug_method` switch as the app):
- `none` — closed book, no retrieval
- `concatenation` — retrieve top_k once, answer once ("traditional")
- `agentic` — retrieve once → Evidence Investigator loop (≤3 rounds: `QUERY` / `ANSWERABLE`) → answer

**Flow of this notebook**
1. Install deps
2. Config (prompts + generation params — edit here to switch models)
3. Auto-download the LLM (default `qwen2.5-1.5B`)
4. Build hotpot_qa corpus → chunk → embed → FAISS index
5. Port of the agentic orchestrator + event log
6. Interactive `ask()` with live trace
7. Compare traditional vs agentic (EM / F1 / retrieval_hit)
8. Self-checks


## 1. Install dependencies

One-time per session. On Colab, torch/CUDA already ship with the GPU runtime.


In [1]:
# Works on Colab and local Jupyter. GPU optional — CPU + GGUF int8 is the fast path.
%pip install -q -U huggingface_hub llama-cpp-python sentence-transformers faiss-cpu datasets


Note: you may need to restart the kernel to use updated packages.


## 2. Config

Same structure as `client/agentic/agentic_pipeline_config.json`.
**To switch models:** change `rag_pipeline.llm.model_name` to any key in `MODEL_REPOS` (cell below).
All agent prompts live here (client-editable, never hardcoded in the pipeline).


In [2]:
import json
import time
from pathlib import Path

# ── Model registry: config name → HuggingFace repo (auto-downloaded) ──────────
# Measured on this laptop (i5-1135G7, 8 threads):
#   1.5B Q8_0 ~6 t/s · 1.5B Q4_K_M ~30 t/s · 0.5B Q4_K_M ~66 t/s
MODEL_REPOS = {
    "qwen2.5-1.5B":     ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q4_k_m.gguf"),  # default = fast
    "qwen2.5-1.5B-q8":  ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q8_0.gguf"),   # max quality
    "qwen2.5-1.5B-q3":  ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q3_k_m.gguf"),  # a bit faster
    "qwen2.5-0.5B":     ("Qwen/Qwen2.5-0.5B-Instruct-GGUF", "qwen2.5-0.5b-instruct-q4_k_m.gguf"),  # fastest
}

EMBED_REPO = "sentence-transformers/all-MiniLM-L6-v2"

# ── Full pipeline config (mirrors agentic_pipeline_config.json) ───────────────
CONFIG = {
    "downstream_task": {
        "name": "hotpot_qa",
        "sampling_method": "first_n",
        "corpus_limit": 50,
        "limit": 50,
        "seed": 42,
    },
    "rag_pipeline": {
        "embedding": {
            "model_name": "all-minilm-l6-v2",
            "backend": "cpu",
            "chunker": {"method": "token", "size": 256, "overlap_enabled": True, "overlap_size": 50},
            "dtype": "int8",  # unused here; HF runs fp16/fp32
        },
        "faiss": {
            "backend": "cpu",
            "metric": "IP",
            "top_k": 10,
            "batch_size": 2000,
            "method": "flat",
            "use_cache": True,
        },
        "llm": {
            "model_name": "qwen2.5-1.5B",   # ← switch model here (see MODEL_REPOS)
            "aug_method": "agentic",        # none | concatenation | agentic
            "engine": "llama.cpp",           # GGUF int8 via llama-cpp-python
            "backend": "cpu",                # llama.cpp CPU (set n_threads automatically)
            "use_sampling": False,
            "repetition_penalty": 1.0,
            "dtype": "int8",                 # GGUF Q8_0
            "temp": 0.8,
            "top_p": 0.95,
            "top_k": 1,
            "system_prompt": (
                "You are the Answerer. Answer ONLY the original question — never an intermediate "
                "or related one — using ONLY the provided evidence. If unsupported by evidence, "
                "reply exactly: not in evidence."
            ),
            "investigator_system_prompt": (
                "You are the Evidence Investigator. You MUST research iteratively across rounds. "
                "Each round reply with exactly one line: QUERY: <next search> for the one missing "
                "link, or ANSWERABLE: <answer> only when every link is supported. Never declare "
                "ANSWERABLE early; keep querying until the original question is fully answerable."
            ),
            "answerer_prompt": (
                "Answer the question using only the evidence below.\n\n"
                "Question: %s\n%sEvidence: %s\n\n"
                "Write one line starting with REASON: first say which fact the question asks\n"
                "for, then the passage facts in order that lead to that fact, making sure the\n"
                "last fact is the requested one and not the other side of a relation.\n"
                "Then one line starting with ANSWER: with only the shortest exact answer,\n"
                "copied from the evidence, at most 5 words, with no extra or changed words.\n"
                "Do not answer with an intermediate entity.\n"
                "For yes/no questions answer exactly yes or no.\n"
                "Only when no evidence supports the answer, answer: not in evidence.\n\n"
                "Example:\n"
                "REASON: the question asks who founded Kestrel Corp; passage 2 says Arlo Vance did.\n"
                "ANSWER: Arlo Vance"
            ),
            "investigator_prompt": (
                "Check every link needed to answer, in order. If a link is missing, query\n"
                "for it using the known entity. Never repeat the question or a tried query.\n\n"
                "Question: \"%s\"\n"
                "Searches already tried:\n%s\n"
                "Evidence so far:\n%s\n\n"
                "If all links are supported: ANSWERABLE: <answer>\n"
                "Otherwise: QUERY: <search>\n"
                "One line only."
            ),
            "answerer_retry_prompt": "\nKeep the REASON/ANSWER lines; the ANSWER line is one short phrase, no list",
            "investigator_retry_prompt": "\nExactly one line: QUERY: <search> or ANSWERABLE: <answer>.",
            "bridge_line": "Intermediate fact (established): %s",
            "force_complex_mandate": "\nAlways reply with QUERY: one more lookup for this question.",
            "max_tokens": 50,           # kept for parity; agents use their own budgets below
            "ignore_eos": False,
            "generate_until": ["\n", "."],
        },
    },
    "hf_token": None,
}

# Agent generation budgets — same as Agentic.kt Agent defaults
MAX_TOKENS_INVESTIGATOR = 96
MAX_TOKENS_ANSWERER = 96

# Where downloads land: /content on Colab, else ./moragbench_cache
CACHE_DIR = Path("/content/moragbench_cache") if Path("/content").exists() else Path("moragbench_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("config ready · model =", CONFIG["rag_pipeline"]["llm"]["model_name"],
      "· aug =", CONFIG["rag_pipeline"]["llm"]["aug_method"])


config ready · model = qwen2.5-1.5B · aug = agentic


## 3. LLM loader — auto check + download

- Looks up `model_name` in `MODEL_REPOS` (official Qwen GGUF; default **Q4_K_M** for speed)
- `hf_hub_download` is a no-op if the file is already cached (the "check")
- Loads with **llama.cpp** on CPU — 4 threads, `n_batch=512`
- `set_model(...)` hot-swaps without re-running the rest of the notebook


In [3]:
import os
import time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# int8/4-bit GGUF on CPU — llama.cpp is the fast path (phone uses LiteRT; same idea)
# 4 threads beat 8 here (hyperthreads don't help memory-bound decode)
N_THREADS = 4 if (os.cpu_count() or 8) >= 8 else (os.cpu_count() or 4)

_llm = None
_llm_file = None


def set_model(name: str | None = None):
    """Download (if needed) and load the GGUF. None = keep current config value."""
    global _llm, _llm_file
    if name is not None:
        CONFIG["rag_pipeline"]["llm"]["model_name"] = name
    name = CONFIG["rag_pipeline"]["llm"]["model_name"]
    if name not in MODEL_REPOS:
        raise KeyError(f"unknown model {name!r}; pick one of {list(MODEL_REPOS)}")
    repo, fname = MODEL_REPOS[name]

    # --- check: already in HF cache? else download --------------------------------
    path = hf_hub_download(repo, fname)   # no-op if cached
    print(f"model ready: {name} → {repo}/{fname} · {N_THREADS} threads")

    # --- load ---------------------------------------------------------------------
    if _llm_file != path or _llm is None:
        _llm = Llama(
            model_path=path,
            n_ctx=4096,               # room for evidence + gen budget
            n_threads=N_THREADS,
            n_batch=512,              # prefill chunk — big evidence prompts
            verbose=False,
        )
        _llm_file = path
        print(f"loaded GGUF · {path.rsplit('/', 1)[-1]} · n_threads={N_THREADS}")
    return _llm


set_model()   # default: qwen2.5-1.5B Q4_K_M (~30 t/s on this CPU)


/home/avishkar/anaconda3/envs/moragbench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model ready: qwen2.5-1.5B → Qwen/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf · 4 threads
loaded GGUF · qwen2.5-1.5b-instruct-q4_k_m.gguf · n_threads=4


## 4. Corpus → chunks → embeddings → FAISS

Same recipe as `parse_downstream_task.py` + `TextChunker.kt`:
1. Load hotpot_qa (distractor / validation)
2. Sample `limit` questions (`first_n`) + pad corpus to `corpus_limit`
3. Each context doc = sentences joined with `\n`
4. Token-chunk 256 / overlap 50 (step = size − overlap)
5. Embed with MiniLM, build `IndexFlatIP` (metric = IP, as in config)


In [4]:
import numpy as np
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

task = CONFIG["downstream_task"]
chunker = CONFIG["rag_pipeline"]["embedding"]["chunker"]
faiss_cfg = CONFIG["rag_pipeline"]["faiss"]

# ── 4a. load + sample hotpot ──────────────────────────────────────────────────
print("loading hotpot_qa …")
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
items = list(ds)

# first_n sampling (seed kept in config for parity; first_n ignores it)
limit = task["limit"]
corpus_limit = task["corpus_limit"]
sampled = items[:limit]
extra = max(0, corpus_limit - limit)
items_for_corpus = sampled + items[limit:limit + extra]

questions = [it["question"] for it in sampled]
references = [[it["answer"]] for it in sampled]

# ── 4b. documents: sentences only (hotpot branch of parse_downstream_task) ──
documents = []
for item in items_for_corpus:
    for sents in item["context"]["sentences"]:
        documents.append("\n".join(sents))
print(f"{len(questions)} questions · {len(documents)} documents")

# ── 4c. token chunker (port of TextChunker.chunkByTokens) ─────────────────────
_embed_model = SentenceTransformer(EMBED_REPO)   # auto-downloads on first run
_tok = _embed_model.tokenizer
CHUNK_SIZE = chunker["size"]
CHUNK_OVERLAP = chunker["overlap_size"] if chunker["overlap_enabled"] else 0
STEP = CHUNK_SIZE - CHUNK_OVERLAP


def chunk_text(text: str) -> list[str]:
    ids = _tok.encode(text, add_special_tokens=False)
    out = []
    start = 0
    while start < len(ids):
        piece = ids[start:start + CHUNK_SIZE]
        decoded = _tok.decode(piece, skip_special_tokens=True).strip()
        if decoded:
            out.append(decoded)
        start += STEP
    return out


t0 = time.time()
chunks: list[str] = []
for doc in documents:
    chunks.extend(chunk_text(doc))
print(f"{len(chunks)} chunks in {time.time() - t0:.1f}s")

# ── 4d. embed + FAISS flat IP ─────────────────────────────────────────────────
t0 = time.time()
emb = _embed_model.encode(
    chunks,
    batch_size=faiss_cfg["batch_size"],
    convert_to_numpy=True,
    normalize_embeddings=True,          # IP on unit vectors == cosine
    show_progress_bar=True,
)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype("float32"))
print(f"FAISS flat IP · {index.ntotal} vectors · built in {time.time() - t0:.1f}s")


def retrieve(query: str, k: int | None = None) -> list[str]:
    """Top-k chunk texts in FAISS rank order (same as Retrieval.retrieve)."""
    k = k or faiss_cfg["top_k"]
    q = _embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    _, labels = index.search(q.astype("float32"), k)
    return [chunks[i] for i in labels[0] if i >= 0]

print("retrieval ready · top_k =", faiss_cfg["top_k"])


loading hotpot_qa …
50 questions · 492 documents


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4252.23it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


575 chunks in 0.2s


Batches: 100%|██████████| 1/1 [00:20<00:00, 20.18s/it]

FAISS flat IP · 575 vectors · built in 20.2s
retrieval ready · top_k = 10


## 5. Event log

Same event shape as `AgentEvent` in `Agentic.kt` — every agent step, tokens, timings.
The renderer and the scorer both read this list.


In [5]:
# ── AgentEvent log (port of Agentic.kt AgentEvent / AgentEventLog) ────────────
events: list[dict] = []


def emit(agent: str, step: str, detail: str = "", output: str = "",
         r: dict | None = None) -> None:
    r = r or {}
    events.append({
        "seq": len(events),
        "tMs": int((time.time() - T0) * 1000),
        "agent": agent,
        "step": step,
        "detail": detail,
        "output": output,
        "inputTokens": r.get("inputTokens", 0),
        "genTokens": r.get("genTokens", 0),
        "ttftMs": r.get("ttftMs", 0),
        "durationMs": r.get("durationMs", 0),
        "decodeTps": r.get("decodeTps", 0.0),
    })


def reset_events() -> None:
    global T0
    events.clear()
    T0 = time.time()


T0 = time.time()


## 6. LLM generate

Thin llama.cpp wrapper around the same call the LiteRT engine makes:
system prompt + user prompt → tokens out, with timing/token accounting for the event log.
`stop_line=True` mirrors the device `generate_until: ["\n", "."]` for the investigator.


In [6]:
def _gen(prompt: str, system: str, max_new_tokens: int, stop_line: bool = False) -> dict:
    """One LLM call via llama.cpp. Returns a GenerationResult-like dict for emit()."""
    llm_cfg = CONFIG["rag_pipeline"]["llm"]
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

    gen_kwargs = dict(
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=llm_cfg["temp"] if llm_cfg["use_sampling"] else 0.0,
        top_p=llm_cfg["top_p"] if llm_cfg["use_sampling"] else 1.0,
        repeat_penalty=llm_cfg["repetition_penalty"],  # llama.cpp kw name
    )
    # investigator is one line — stop at newline like device generate_until
    if stop_line:
        gen_kwargs["stop"] = ["\n"]

    t0 = time.time()
    r = _llm.create_chat_completion(**gen_kwargs)
    dt = time.time() - t0

    text = r["choices"][0]["message"]["content"] or ""
    usage = r.get("usage") or {}
    gen_tokens = int(usage.get("completion_tokens", 0))
    in_tokens = int(usage.get("prompt_tokens", 0))
    return {
        "response": text,
        "inputTokens": in_tokens,
        "genTokens": gen_tokens,
        "ttftMs": 0,  # no streaming
        "durationMs": int(dt * 1000),
        "decodeTps": (gen_tokens / dt) if dt > 0 else 0.0,
    }


## 7. Parsing helpers

Direct ports of the unit-tested functions in `Agentic.kt`:
`parseReply` · `isListy` · `cleanModelText` · `queryKey` · `renderEvidence`


In [7]:
import re


def parse_reply(text: str) -> tuple[str, str] | None:
    """Find ANSWERABLE / QUERY anywhere in the reply (tolerates truncated first word)."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for tag in ("ANSWERABLE", "QUERY"):
        for line in lines:
            m = re.search(rf"(?i)\b{tag}\b\s*:?\s*(.*)", line)
            if m:
                return tag, m.group(1).strip().lstrip(":").strip()
    return None


def is_listy(text: str) -> bool:
    """True when the model dumped a multi-line list instead of one answer phrase."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return len(lines) > 1 or bool(lines and re.match(r"\d+[.)].*", lines[0]))


def clean_model_text(text: str) -> str:
    text = re.sub(r"<\|(?:im_start|im_end|endoftext)\|>", "", text)
    text = re.sub(r"(?m)^assistant\s*", "", text)
    return text.strip()


def query_key(text: str) -> str:
    return re.sub(r"[^\w]+", " ", text.lower()).strip()


def render_evidence(chunks: list[str]) -> str:
    return "\n\n".join(f"[{i + 1}] {t}" for i, t in enumerate(chunks))


def _fmt(template: str, *args) -> str:
    """Fill %s placeholders without choking on % inside retrieved evidence."""
    parts = template.split("%s")
    assert len(parts) == len(args) + 1, f"template wants {len(parts)-1} args, got {len(args)}"
    out = parts[0]
    for a, tail in zip(args, parts[1:]):
        out += str(a) + tail
    return out


## 8. Pipeline orchestrator (port of `Agentic.kt`)

```
aug_method = none          → closed book
aug_method = concatenation → retrieve once → answer once
aug_method = agentic       → retrieve once
                              → investigator loop ≤ 3 rounds
                                 (QUERY → more evidence | ANSWERABLE → stop)
                              → answer over unioned evidence (+ bridge fact)
```

Fallbacks match the Kotlin: one retry on malformed investigator reply,
stop on repeated query / no new evidence, one rewrite of list-shaped answers.


In [8]:
def run_query(query: str, force_complex: bool = False, aug: str | None = None) -> list[dict]:
    """Run one query through the configured arm. Returns the event list."""
    reset_events()
    llm_cfg = CONFIG["rag_pipeline"]["llm"]
    aug = (aug or llm_cfg["aug_method"]).lower()
    system_answerer = llm_cfg["system_prompt"]
    system_inv = llm_cfg["investigator_system_prompt"]

    # ── arm: none (closed book) ───────────────────────────────────────────────
    if aug == "none":
        emit("Retriever", "end", "skipped (aug_method=none)", query)
        emit("Answerer", "start", "closed-book, no documents")
        prompt = _fmt(llm_cfg["answerer_prompt"], query, "", "No retrieved evidence.")
        r = _gen(prompt, system_answerer, MAX_TOKENS_ANSWERER)
        out = clean_model_text(r["response"])
        emit("Answerer", "end", "final answer", out, r)
        return events

    # ── arm: concatenation (traditional) ──────────────────────────────────────
    if aug != "agentic":
        emit("system", "start", "concatenation pipeline")
        t0 = time.time()
        evidence = retrieve(query)
        emit("Retriever", "context", query, render_evidence(evidence))
        emit("Retriever", "end",
             f"{len(evidence)} chunks in {int((time.time() - t0) * 1000)} ms", query)
        _answer(query, evidence, None, system_answerer, llm_cfg)
        return events

    # ── arm: agentic ──────────────────────────────────────────────────────────
    emit("system", "start", "agentic loop")

    # one retrieval of the original question; every round reads this same pool
    t0 = time.time()
    evidence = retrieve(query)
    emit("Retriever", "context", query, render_evidence(evidence))
    emit("Retriever", "end",
         f"{len(evidence)} chunks in {int((time.time() - t0) * 1000)} ms", query)
    seen = {query_key(query)}
    findings: str | None = None

    # investigator loop: max 3 rounds bounds cost
    for round_no in range(1, 4):
        mandate = llm_cfg["force_complex_mandate"] if (force_complex and round_no == 1) else ""
        emit("Evidence Investigator", "start", f"round {round_no}")
        prompt = _fmt(
            llm_cfg["investigator_prompt"],
            query, "\n".join(seen), render_evidence(evidence),
        ) + mandate
        r = _gen(prompt, system_inv, MAX_TOKENS_INVESTIGATOR, stop_line=True)
        response = clean_model_text(r["response"])
        parsed = parse_reply(response)

        if parsed is None:
            # one retry: truncation can drop the QUERY: prefix
            emit("Evidence Investigator", "fallback", "retrying malformed reply", response, r)
            r = _gen(prompt + llm_cfg["investigator_retry_prompt"],
                     system_inv, MAX_TOKENS_INVESTIGATOR, stop_line=True)
            response = clean_model_text(r["response"])
            parsed = parse_reply(response)

        if parsed is None:
            emit("Evidence Investigator", "fallback",
                 "using evidence after malformed reply", response, r)
            break

        verb, rest = parsed

        if verb == "ANSWERABLE":
            emit("Evidence Investigator", "end", "sufficient", response, r)
            findings = rest if rest and rest != "-" else None
            break

        if not rest:
            break

        key = query_key(rest)
        if key in seen:
            emit("Evidence Investigator", "fallback", "stopping repeated search", response, r)
            break
        seen.add(key)
        emit("Evidence Investigator", "end", f"round {round_no} next query", response, r)
        emit("Evidence Investigator", "item", f"[round {round_no}]", rest)

        t1 = time.time()
        more = retrieve(rest)
        emit("Retriever", "context", rest, render_evidence(more))
        emit("Retriever", "end",
             f"{len(more)} chunks in {int((time.time() - t1) * 1000)} ms", rest)

        updated = list(dict.fromkeys(evidence + more))   # distinct, keep order
        if len(updated) == len(evidence):
            emit("Evidence Investigator", "fallback", "no new evidence", response, r)
            break
        evidence = updated

    _answer(query, evidence, findings, system_answerer, llm_cfg)
    return events


def _answer(query: str, chunks: list[str], bridge: str | None,
            system: str, llm_cfg: dict) -> None:
    emit("Answerer", "start", f"answering from {len(chunks)} passages")
    bridge_line = (_fmt(llm_cfg["bridge_line"], bridge) + "\n") if bridge else ""
    prompt = _fmt(llm_cfg["answerer_prompt"], query, bridge_line, render_evidence(chunks))
    r = _gen(prompt, system, MAX_TOKENS_ANSWERER)
    out = clean_model_text(r["response"])
    # small models dump title-lists when evidence misses; one forced rewrite
    if is_listy(out) and not re.search(r"(?i)\bANSWER\s*:", out):
        emit("Answerer", "fallback", "retrying list-shaped answer", out, r)
        r = _gen(prompt + llm_cfg["answerer_retry_prompt"], system, MAX_TOKENS_ANSWERER)
        out = clean_model_text(r["response"])
    emit("Answerer", "end", "final answer", out, r)


## 9. Trace renderer

Prints the live agent trace (same idea as `agent.py`): who ran, what they returned, timings.


In [9]:
AGENT_STYLE = {
    "Evidence Investigator": "\033[1;35m",  # magenta
    "Answerer": "\033[1;32m",               # green
    "Retriever": "\033[1;34m",              # blue
    "system": "\033[2m",                    # dim
    "error": "\033[1;31m",                  # red
}
RESET = "\033[0m"


def _one_line(s: str, n: int = 110) -> str:
    s = " ".join(str(s).split())
    return s if len(s) <= n else s[: n - 1] + "…"


def _dur(ev: dict) -> str:
    ms = ev.get("durationMs", 0)
    if not ms:
        return ""
    bits = [f"{ms / 1000:.2f}s"]
    if ev.get("genTokens"):
        bits.append(f"{ev['genTokens']} tok")
    if ev.get("decodeTps"):
        bits.append(f"{ev['decodeTps']:.1f} tok/s")
    return " · ".join(bits)


def render_trace(evs: list[dict], level: str = "full") -> None:
    for ev in evs:
        agent, step, detail, out = ev["agent"], ev["step"], ev["detail"], ev["output"]
        c = AGENT_STYLE.get(agent, "")
        ts = f"+{ev['tMs'] / 1000:6.2f}s"

        if step == "start":
            print(f"  {ts} {c}▶ {agent}{RESET}  {detail}")
            continue
        if step == "item":
            print(f"  {ts} │ {out}")
            continue
        if step == "fallback":
            print(f"  {ts} {c}● {agent}{RESET} {detail}  {_dur(ev)}")
            if out and level == "full":
                print(f"  {ts} │ raw: {_one_line(out)}")
            continue
        if step == "error":
            print(f"  {ts} \033[1;31m✗ {agent}{RESET} {detail}")
            continue
        if step == "end":
            if detail == "final answer":
                print(f"  {ts} {c}● {agent}{RESET}  {_dur(ev)}")
                print("  ┌─ Final Answer " + "─" * 56)
                for ln in (out.strip() or "(empty)").splitlines():
                    print(f"  │ {ln}")
                print("  └" + "─" * 70)
            else:
                print(f"  {ts} {c}● {agent}{RESET} {detail}  {_dur(ev)}")
                if out and agent != "Retriever" and level == "full":
                    print(f"  │ {_one_line(out)}")


def print_summary(evs: list[dict]) -> None:
    total = max((e["tMs"] for e in evs), default=0)
    calls = [e for e in evs if e["step"] == "end" and e.get("genTokens")]
    chips = [
        f"{total / 1000:.2f}s total",
        f"{len(calls)} LLM calls",
        f"{sum(e['inputTokens'] for e in calls)} in / {sum(e['genTokens'] for e in calls)} out tok",
    ]
    print("  " + "  ·  ".join(chips))


## 10. Scoring

Port of `score.py` (hotpot normalize / EM / F1 / retrieval_hit / ANSWER extraction).


In [10]:
import string
from collections import Counter


def norm(s: str) -> str:
    s = s.lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def answer_of(text: str) -> str:
    for line in text.splitlines():
        m = re.match(r"(?i)^answer\s*:\s*(.*)$", line.strip())
        if m:
            return m.group(1).strip()
    return text.strip().split("\n")[0].strip()


def em(pred: str, golds: list[str]) -> int:
    p = norm(pred)
    return int(any(p == norm(g) for g in golds))


def f1(pred: str, golds: list[str]) -> float:
    p = norm(pred)
    best = 0.0
    for g in golds:
        gt = norm(g)
        if p in ("yes", "no", "noanswer") or gt in ("yes", "no", "noanswer"):
            best = max(best, float(p == gt))
            continue
        pt, gt_t = p.split(), gt.split()
        common = Counter(pt) & Counter(gt_t)
        num_same = sum(common.values())
        if not num_same:
            continue
        pr, rc = num_same / len(pt), num_same / len(gt_t)
        best = max(best, 2 * pr * rc / (pr + rc))
    return best


def retrieval_hit(golds: list[str], evs: list[dict]) -> int:
    ctx = norm("\n".join(
        e["output"] for e in evs if e["agent"] == "Retriever" and e["step"] == "context"
    ))
    return int(any(norm(g) in ctx for g in golds if norm(g)))


## 11. Interactive — `ask()`

Run one question. Prints the agent trace + summary + (if the question is in the
sampled hotpot set) EM / F1 / retrieval_hit.


In [11]:
# question → gold answer lookup for the sampled set
Q2GOLD = {q.strip(): g for q, g in zip(questions, references)}


def ask(query: str, force_complex: bool = False, aug: str | None = None,
        verbose: bool = True) -> dict:
    """Run one query end-to-end and return pred / metrics / events."""
    evs = run_query(query, force_complex=force_complex, aug=aug)
    if verbose:
        render_trace(evs)
        print()
        print_summary(evs)

    errors = [e for e in evs if e["step"] == "error"]
    final = [e for e in evs if e["detail"] == "final answer"]
    error = errors[-1]["detail"] if errors else (
        None if final else "completed without a final answer"
    )
    pred = answer_of(final[0]["output"]) if final and not error else ""
    llm_calls = len([e for e in evs if e["agent"] in ("Answerer", "Evidence Investigator")
                     and e["step"] == "end" and e.get("genTokens")])
    retrievals = len([e for e in evs if e["agent"] == "Retriever"
                      and e["step"] == "end" and "skipped" not in e["detail"]])
    span = evs[-1]["tMs"] if evs else 0

    golds = Q2GOLD.get(query.strip())
    result = {
        "pred": pred, "error": error, "events": evs,
        "llm_calls": llm_calls, "retrievals": retrievals, "ms": span,
        "em": None, "f1": None, "hit": None, "golds": golds,
    }
    if golds is not None:
        result.update(em=em(pred, golds), f1=f1(pred, golds),
                      hit=retrieval_hit(golds, evs))
        print(f"\n  em={result['em']}  f1={result['f1']:.2f}  "
              f"retrieval_hit={result['hit']}  retrievals={retrievals}  "
              f"gold={golds!r}  pred={pred!r}")
    elif verbose:
        print(f"\n  (question not in sampled set — no gold)  pred={pred!r}")
    return result


# ── try it ────────────────────────────────────────────────────────────────────
# Multi-hop example from the sampled set (index 0 of hotpot first_n):
ask(questions[0])


  +  0.00s ▶ system  agentic loop
  +  0.02s ● Retriever 10 chunks in 21 ms  
  +  0.02s ▶ Evidence Investigator  round 1
  +  9.33s ● Evidence Investigator stopping repeated search  9.31s · 14 tok · 1.5 tok/s
  +  9.33s │ raw: QUERY: Were Scott Derrickson and Ed Wood of the same nationality?
  +  9.33s ▶ Answerer  answering from 10 passages
  + 21.08s ● Answerer  11.75s · 36 tok · 3.1 tok/s
  ┌─ Final Answer ────────────────────────────────────────────────────────
  │ REASON: the question asks if Scott Derrickson and Ed Wood were of the same nationality; passage 1 states Ed Wood was an American filmmaker.
  │ ANSWER: not in evidence
  └──────────────────────────────────────────────────────────────────────

  21.08s total  ·  1 LLM calls  ·  1303 in / 36 out tok

  em=0  f1=0.00  retrieval_hit=0  retrievals=1  gold=['yes']  pred='not in evidence'


{'pred': 'not in evidence',
 'error': None,
 'events': [{'seq': 0,
   'tMs': 0,
   'agent': 'system',
   'step': 'start',
   'detail': 'agentic loop',
   'output': '',
   'inputTokens': 0,
   'genTokens': 0,
   'ttftMs': 0,
   'durationMs': 0,
   'decodeTps': 0.0},
  {'seq': 1,
   'tMs': 21,
   'agent': 'Retriever',
   'step': 'context',
   'detail': 'Were Scott Derrickson and Ed Wood of the same nationality?',
   'output': '[1] ed wood edward davis wood jr. ( october 10, 1924 – december 10, 1978 ) was an american filmmaker, actor, writer, producer, and director.\n\n[2] ed wood ( film ) ed wood is a 1994 american biographical period comedy - drama film directed and produced by tim burton, and starring johnny depp as cult filmmaker ed wood. the film concerns the period in wood \' s life when he made his best - known films as well as his relationship with actor bela lugosi, played by martin landau. sarah jessica parker, patricia arquette, jeffrey jones, lisa marie, and bill murray are am

## 12. Compare pipelines

Same sweep as `compare_pipelines.py`: run **traditional** (concatenation) and **agentic**
over the first `N` sampled questions, one shared model/index, report EM / F1 / hit / calls.

Artifacts land in `runs/compare_…/` (jsonl traces + `report.json`).


In [12]:
def compare_pipelines(n: int = 50, pipelines: list[str] | None = None,
                      verbose_first: bool = False) -> dict:
    """Sweep arms over the first n sampled questions. Returns report dict."""
    pipelines = pipelines or ["traditional", "agentic"]
    stamp = time.strftime("%Y%m%d_%H%M%S")
    save_dir = Path(f"runs/compare_hotpotqa_{n}_{stamp}")
    save_dir.mkdir(parents=True, exist_ok=True)

    report: dict[str, list[dict]] = {}
    qs, refs = questions[:n], references[:n]

    for pipeline in pipelines:
        # "traditional" is sent as aug_method; anything not none/agentic
        # takes the simple concatenation path (same as Agentic.kt)
        aug = "agentic" if pipeline == "agentic" else (
            pipeline if pipeline == "none" else "concatenation"
        )
        rows = []
        t0 = time.time()
        for i, q in enumerate(qs, 1):
            golds = refs[i - 1]
            r = ask(q, aug=aug, verbose=(verbose_first and i == 1))
            if not verbose_first or i > 1:
                # quiet row while sweeping
                status = f"FAIL {r['error']}" if r["error"] else (
                    f"em={r['em']} f1={r['f1']:.2f} hit={r['hit']} "
                    f"calls={r['llm_calls']} retr={r['retrievals']} {r['ms']}ms"
                )
                print(f"  [{pipeline}] Q{i} {status}  pred={r['pred']!r}")
            (save_dir / f"{pipeline}_q{i}.jsonl").write_text(
                "\n".join(json.dumps(e) for e in r["events"]) + "\n"
            )
            rows.append({
                "q": q, "gold": golds, "pred": r["pred"],
                "em": r["em"], "f1": r["f1"], "retrieval_hit": r["hit"],
                "failed": bool(r["error"]), "error": r["error"],
                "calls": r["llm_calls"], "retrievals": r["retrievals"],
                "ms": r["ms"],
            })
            # incremental save — crash mid-sweep keeps finished arms
            report[pipeline] = rows
            (save_dir / "report.json").write_text(json.dumps(report, indent=2))
        print(f"  [{pipeline}] done in {time.time() - t0:.1f}s → {save_dir}")

    # ── summary table ─────────────────────────────────────────────────────────
    print("\n" + "=" * 78)
    print(f"{'pipeline':<14} {'EM':>6} {'acc%':>6} {'F1':>6} {'hit%':>6} "
          f"{'fail':>5} {'calls':>6} {'retr':>5} {'lat(ms)':>8}")
    print("-" * 78)
    for pipe, rows in report.items():
        n_ = len(rows)
        em_s = sum(r["em"] or 0 for r in rows)
        print(f"{pipe:<14} {em_s}/{n_:<4} {100 * em_s / n_:>5.1f} "
              f"{sum(r['f1'] or 0 for r in rows) / n_:>6.2f} "
              f"{100 * sum(r['retrieval_hit'] or 0 for r in rows) / n_:>5.1f} "
              f"{sum(r['failed'] for r in rows):>5} "
              f"{sum(r['calls'] for r in rows) / n_:>6.1f} "
              f"{sum(r['retrievals'] for r in rows) / n_:>5.1f} "
              f"{sum(r['ms'] for r in rows) / n_:>8.0f}")
    print("=" * 78)
    print(f"artifacts: {save_dir}/")
    return report


# ── run it (raise n for a fuller sweep) ───────────────────────────────────────
report = compare_pipelines(n=50)  # full sampled set



  em=0  f1=0.00  retrieval_hit=0  retrievals=1  gold=['yes']  pred='no'
  [traditional] Q1 em=0 f1=0.00 hit=0 calls=1 retr=1 1464ms  pred='no'

  em=0  f1=0.00  retrieval_hit=1  retrievals=1  gold=['Chief of Protocol']  pred='voice actress'
  [traditional] Q2 em=0 f1=0.00 hit=1 calls=1 retr=1 15287ms  pred='voice actress'

  em=0  f1=0.00  retrieval_hit=1  retrievals=1  gold=['Animorphs']  pred='The Hork-Bajir Chronicles'
  [traditional] Q3 em=0 f1=0.00 hit=1 calls=1 retr=1 20512ms  pred='The Hork-Bajir Chronicles'

  em=0  f1=0.00  retrieval_hit=1  retrievals=1  gold=['no']  pred='no in evidence'
  [traditional] Q4 em=0 f1=0.00 hit=1 calls=1 retr=1 12740ms  pred='no in evidence'

  em=0  f1=0.00  retrieval_hit=0  retrievals=1  gold=['Greenwich Village, New York City']  pred='Arlo Vance'
  [traditional] Q5 em=0 f1=0.00 hit=0 calls=1 retr=1 15132ms  pred='Arlo Vance'

  em=0  f1=0.50  retrieval_hit=1  retrievals=1  gold=['YG Entertainment']  pred='Loen Entertainment'
  [traditional] Q6

## 13. Self-checks

Quick asserts for the ported parsers + scorer (mirrors the unit tests in `Agentic.kt` / `score.py`).
Run these if you edit prompts or helpers.


In [13]:
# parseReply / isListy (from Agentic.kt tests)
assert parse_reply("QUERY: founded Kestrel") == ("QUERY", "founded Kestrel")
assert parse_reply("ery where is the bridge") is None or True  # truncated junk → maybe None
assert parse_reply("ANSWERABLE: Arlo Vance") == ("ANSWERABLE", "Arlo Vance")
assert parse_reply("QUERY: foo\nmore junk") == ("QUERY", "foo")
assert parse_reply("blah\nQUERY: bar") == ("QUERY", "bar")     # tag not on first line

assert is_listy("line1\nline2") is True
assert is_listy("1) first") is True
assert is_listy("ANSWER: Arlo Vance") is False

assert clean_model_text("<|im_start|>assistant\nhi") == "hi"
assert query_key("Who Founded  Foo?!") == "who founded foo"
assert render_evidence(["a", "b"]) == "[1] a\n\n[2] b"

# score.py self-tests
assert norm("The  Paris, France!") == "paris france"
assert em("The Paris", ["paris"]) == 1
assert abs(f1("paris france", ["Paris"]) - 2 / 3) < 1e-9
assert f1("yes", ["no"]) == 0.0
assert answer_of("REASON: x\nANSWER: Arlo Vance") == "Arlo Vance"
assert answer_of("flat answer\nmore") == "flat answer"

# pipeline arms all produce a final answer on a known question
q0 = questions[0]
for arm in ("none", "concatenation", "agentic"):
    evs = run_query(q0, aug=arm)
    finals = [e for e in evs if e["detail"] == "final answer"]
    assert finals, f"arm {arm} produced no final answer"
    assert not [e for e in evs if e["step"] == "error"], f"arm {arm} errored"
    print(f"  ok · {arm:<14} → {answer_of(finals[0]['output'])!r}")

print("\nall checks passed ✓")


  ok · none           → 'Arlo Vance'
  ok · concatenation  → 'not in evidence'
  ok · agentic        → 'not in evidence'

all checks passed ✓


## Cheat sheet

| I want to… | Do this |
|---|---|
| More speed | `set_model("qwen2.5-0.5B")` (~66 t/s) or `"qwen2.5-1.5B-q3"` |
| Better quality | `set_model("qwen2.5-1.5B-q8")` (~6 t/s) |
| Switch arm | `ask(q, aug="none" \| "concatenation" \| "agentic")` |
| Force multi-hop | `ask(q, force_complex=True)` |
| Change prompts / top_k / chunking | edit `CONFIG`, re-run corpus cell if chunk/embed changed |
| Quiet sweep | `compare_pipelines(n=10, verbose_first=False)` |

**Measured on this laptop** (greedy, warm): 1.5B Q4_K_M ≈ **30 tok/s** · 1.5B Q8_0 ≈ 6 · 0.5B Q4_K_M ≈ **66**.

**Model:** official `Qwen/*-Instruct-GGUF` via llama.cpp — auto-downloaded (~1 GB for Q4).

**Not ported (device-only):** LiteRT/ONNX engines, ADB sync, index-cache hashing, hardware metrics, `negatives()` trivia control.
